# Lesson 84 · LoRA — sticky notes for a big model: W = W₀ + BA fine-tunes a GPT-style model with 0.1 % of the parameters

**Learning goals**: implement `LoRALinear`, and see why the low-rank patch $\Delta W=BA$ touches only a handful of parameters yet can still pull the model toward a new task.

> Links back to low rank in L14; the picture: freeze $W_0$, train only the small matrices $B,A$.

🔗 **Aurora connection**: `LoRALinear` will plug into Aurora's `src/aurora/llm/` (LLM fine-tuning) and `src/aurora/speech/` (speech fine-tuning). Right now `src/aurora/llm/` already has `kvcache.py` (the topic of L85), `retrieve.py` and `sample.py`; `finetune.py` is a planned module — speech fine-tuning will live in `aurora.speech` (there is no `aurora.asr` sub-package, so don't reference one).


← **Previous lesson** · [L83 · A Transformer from scratch](https://github.com/djzoom/AURORA/blob/main/notebooks/9_llm/L83_transformer.ipynb) *(Chinese)*

> Last lesson covered **a Transformer from scratch**: a complete implementation of multi-head attention + positional encoding + feed-forward.
> This lesson explores **LoRA**, low-rank adaptation.

## How can LoRA give a model a "new personality" with just 0.1 % of the parameters?

Picture this: you own a printed encyclopedia (the pretrained model, 7B parameters). You can't reprint it, but you can **stick on sticky notes** — one thin note per page with the corrections written on it.

**That is the idea behind LoRA (Low-Rank Adaptation)**: leave the original weights W₀ alone, and wire a low-rank correction matrix ΔW in parallel beside them.

**The low-rank intuition**: a 512×512 matrix has 262,144 parameters. But if you approximate it by the product of two thin matrices B (512×8) and A (8×512), you need only 8,192 parameters — a 97 % saving. Can that possibly work? It can, because **real-world "task changes" are usually low-dimensional** — switching from translating into English to translating into French changes only a few "directions", not the whole space of knowledge.

**Forward pass**:

```
h = W₀x + ΔWx = W₀x + (B·A)x
```

- **W₀**: frozen — first, keep the general-purpose ability safely locked in
- **A** (r×d): initialized from N(0, 0.01) — brings in randomness
- **B** (d×r): initialized to **zero** — so training starts out barely disturbing the original model

This section's task: write `LoRALinear(nn.Module)`, verify the parameter-saving formula, and watch for yourself what rank does on a "keyword spotting" task.

## ⚠️ Common misconceptions

> Don't read "low rank" as "compress W₀ into a low-rank approximation". W₀ itself is usually close to full rank, and forcing it to low rank would destroy a lot of pretrained knowledge. LoRA's low-rank assumption is about the **fine-tuning increment ΔW**: the change from "general model" to "specific task" usually runs along only a few directions, so ΔW can be expressed as the low-rank B·A while W₀ keeps every single number.
>
> Two easy mistakes follow from this. **Bigger rank is not always better** — a large r doesn't just add parameters; it stuffs noise directions into a low-dimensional space of task changes, and on small data it actually overfits more easily (you'll see this yourself in this lesson's parameter experiment). **Freezing W₀ is not only about saving memory** — it keeps the pretrained knowledge from being washed away by gradients from a small dataset (avoiding catastrophic forgetting), and it lets one base model carry several sets of LoRA sticky notes in parallel, plugged in and out per task.

In [ ]:
import torch
import torch.nn as nn

## 1. The core LoRA formula

Forward pass of the original linear layer: `h = W_pretrained @ x`

With LoRA added: `h = (W_pretrained + B @ A) @ x`

where:
- `W_pretrained` has shape `(d_out, d_in)` and is **frozen — it takes no gradient**
- `A` has shape `(r, d_in)`, with `r << min(d_in, d_out)`
- `B` has shape `(d_out, r)`
- The only learnable parameters are `A` and `B`, `r * (d_in + d_out)` in total

In [ ]:
# A quick demo: the rank bound of a low-rank matrix
d_in, d_out, r = 512, 512, 8
A = torch.randn(r, d_in)
B = torch.zeros(d_out, r)
delta_W = B @ A
print(f'delta_W shape: {delta_W.shape}')
print(f'rank of delta_W (in theory <= r={r}): {torch.linalg.matrix_rank(delta_W)}')

# Check the rank when B is nonzero
B2 = torch.randn(d_out, r)
delta_W2 = B2 @ A
print(f'rank of delta_W with random B: {torch.linalg.matrix_rank(delta_W2)} (<=r={r})')

## 2. Initialization strategy: B=0, A~N(0, 0.01)

**Why initialize B to zero?** At the start of training `B @ A = 0`, i.e. `ΔW = 0`, so the model's output is exactly the same as with the original pretrained weights. That guarantees a "no-disturbance start" for fine-tuning — if B were random, the very first outputs would carry a random offset that damages the pretrained knowledge.

**Why a random Gaussian for A?** We need to break the symmetry so that the rows of `A` explore different directions; if every row were identical, it would be equivalent to `r=1`.

**The scaling factor alpha**: the actual output adds `(alpha/r) * B @ A @ x`. `alpha` is a hyperparameter, often set equal to `r` (an effective scale of 1); you can also turn it up to strengthen LoRA's influence.

In [ ]:
# Show that delta_W really is zero after initialization
r, d = 8, 64
lora_A = nn.Parameter(torch.randn(r, d) * 0.01)
lora_B = nn.Parameter(torch.zeros(d, r))
delta = lora_B @ lora_A
print(f'every element of delta_W is zero after init: {torch.allclose(delta, torch.zeros_like(delta))} ✅')

# After one gradient step, B is no longer zero
x_demo = torch.randn(4, d)
out = x_demo @ lora_A.T @ lora_B.T
loss = out.sum()
loss.backward()
print(f'gradient of B is nonzero (|grad|={lora_B.grad.abs().mean():.4f}), so after an update LoRA contributes ✅')

## 3. Parameter efficiency: 97 % fewer trainable parameters

Take `d=512, r=8` as an example:

```
original weight W: 512 × 512 = 262,144 params  (frozen)
LoRA A:              8 × 512 =   4,096 params  (trainable)
LoRA B:            512 × 8   =   4,096 params  (trainable)
total trainable:                 8,192 params
saving: 1 - 8192/262144 = 96.9%
```

A real LLM has hundreds of linear layers. With LoRA on every one of them, the trainable parameters usually come to only 0.1 %–1 % of the model's total, and video memory (VRAM) and compute drop dramatically.

In [ ]:
# Parameter-count comparison
def count_params(params_iter):
    return sum(p.numel() for p in params_iter)

d = 512
# bias=False: matches the worked example above (512×512 = 262,144, no bias).
# With a bias, nn.Linear(d, d) has d=512 extra parameters (262,656 in total).
base = nn.Linear(d, d, bias=False)
total = count_params(base.parameters())
print(f'full linear layer params (no bias): {total:,}')

for r in [4, 8, 16, 32]:
    lora_params = r * d + d * r  # A + B
    saving = 1 - lora_params / total
    print(f'LoRA r={r:2d}: trainable {lora_params:6,}  saving {saving:.1%}')

## 4. ✏️ Implement `class LoRALinear(nn.Module)`

**Line of reasoning**:

```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        ...
    def forward(self, x):
        # h = base(x) + scale * lora_B(lora_A(x))
```

**Three steps to implement it**:

| Step | Action | Key point |
|---|---|---|
| 1 | `__init__`: create `base=nn.Linear(..., bias=False)` + `lora_A`/`lora_B`; freeze base | `base.weight.requires_grad=False`; initialize `lora_B` to zero (`nn.init.zeros_`) |
| 2 | `forward`: `h = base(x) + scale * lora_B(lora_A(x))` | `scale = alpha / rank` |
| 3 | `trainable_params`: count only parameters with `requires_grad=True` | confirms base is properly frozen |

**Acceptance criteria**:
- trainable parameters = `in_features * rank + rank * out_features` (the two matrices A + B)
- at the start of training `base(x) ≈ lora(x)` (B=0 guarantees ΔW=0)
- `base.weight.requires_grad` must be `False`

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=1.0):
        super().__init__()
        # ✏️ TODO: create self.base (nn.Linear) and freeze it
        # ✏️ TODO: create self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        # ✏️ TODO: create self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        # ✏️ TODO: self.scale = alpha / rank
        raise NotImplementedError

    def forward(self, x):
        # ✏️ TODO: base_out = self.base(x)
        # ✏️ TODO: lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        # ✏️ TODO: return base_out + self.scale * lora_out
        raise NotImplementedError

In [ ]:
try:
    # Check 1: number of trainable parameters
    l = LoRALinear(64, 64, rank=4)
    trainable = sum(p.numel() for p in l.parameters() if p.requires_grad)
    expected = 2 * 64 * 4
    assert trainable == expected, f'expected {expected}, got {trainable}'
    print(f'✅ trainable params: {trainable} == 2×64×4')

    # Check 2: forward output has the right shape
    x = torch.randn(3, 64)   # batch=3
    y = l(x)
    assert y.shape == (3, 64), f'wrong output shape: {y.shape}'
    print(f'✅ output shape: {y.shape}')

    # Check 3: the LoRA increment is zero at init (B=0 → lora_out=0)
    lora_out = (x @ l.lora_A.T) @ l.lora_B.T
    assert torch.allclose(lora_out, torch.zeros_like(lora_out))
    print('✅ LoRA increment is zero at init, so the pretrained output is undisturbed')

    # Check 4: 3-D input (batch, seq, d) is handled too
    x3d = torch.randn(2, 5, 64)
    y3d = l(x3d)
    assert y3d.shape == (2, 5, 64)
    print(f'✅ 3-D input shape: {x3d.shape} → output {y3d.shape}')
    # Check 5: scale factor is correct (alpha / rank)
    l_scale = LoRALinear(64, 64, rank=4, alpha=8.0)
    assert abs(l_scale.scale - 2.0) < 1e-6, f'expected scale 2.0, got {l_scale.scale}'
    print(f'✅ scale = alpha/rank = 8.0/4 = {l_scale.scale}')

except (NotImplementedError, TypeError):
    print('⬜ not implemented yet')

## 5. Parameter experiment: how rank affects fine-tuning

Below we simulate a tiny "keyword spotting" task: fine-tune a d→d→1 network with LoRA and compare different settings.

With `d=64` (the dimension the experiment actually uses):

| Setting | LoRA trainable params | Share of the full d×d | Observed (seed=42) |
|------|----------------|---------------------|----------|
| Full-parameter fine-tuning | d²+d ≈ 4,160 | 100% | accuracy ≈ 0.97 |
| LoRA `rank=4` | 2×d×4 = 512 | ~12.5% | accuracy ≈ 0.97, **no worse than full fine-tuning** |
| LoRA `rank=16` | 2×d×16 = 2,048 | ~50% | accuracy ≈ 0.97, almost no different from rank=4 |

> **Note**: the "~6 %" you often see in textbooks is for d=512, rank=16 (2×512×16 / 512² ≈ 6.25%).
> This experiment uses d=64 to keep the demo fast; there rank=16 is about 50 %. It shows the same *proportional relationship*, not industrial-scale absolute numbers.

**Observed results (reproducible with `torch.manual_seed(42)`)**: all three land around 0.97 accuracy, within 1 % of each other,
and `rank=4` is often the best of the three — **on this toy task, rank barely matters**.

**Why?** Two structural reasons:

1. **All three variants have a trainable final layer `nn.Linear(d, 1)`**. The rule for this task is `x[:, 0] > 0` —
   linearly separable in input space. The first layer (whether fully trained or a frozen random `base`) is a roughly
   information-preserving linear map, so the information in `x[:, 0]` is still linearly present in its output,
   and **the final layer's 65 parameters alone can almost solve the task**.
2. **The ΔW the task needs is itself low rank** (close to rank 0, even — the base needs almost no correction),
   so rank=4 already has more than enough expressive power, and going to rank=16 naturally gains nothing.

**The real lesson**: don't expect "bigger rank, better results" to be a universal law. Rank only matters
when *the weight correction ΔW the task needs is itself high rank*; on simple or low-rank tasks a small rank loses nothing —
which is exactly why the LoRA paper dared to fine-tune GPT-3 with r=1–8 (the ablation in §7 of the paper also finds r=1 is often enough).


In [ ]:
torch.manual_seed(42)
d = 64  # feature dimension
steps = 300
lr = 1e-2

def run_experiment(label, model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        x = torch.randn(64, d)
        y = (x[:, 0] > 0).float()   # rule: first dimension positive → class 1
        pred = model(x).squeeze(-1)
        loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = torch.randn(1024, d)
        y = (x[:, 0] > 0).float()
        pred = model(x).squeeze(-1)
        acc = ((pred > 0) == y.bool()).float().mean().item()
    print(f'{label:25s}  trainable params: {trainable:6,}  accuracy: {acc:.3f}')

try:
    # Full parameters: d→d (all trainable) + d→1, same depth as the LoRA variants
    full = nn.Sequential(nn.Linear(d, d), nn.Linear(d, 1))
    run_experiment('Full fine-tuning', full)

    # LoRA rank=4: frozen base d→d, train only the low-rank part + the final d→1
    lora4 = nn.Sequential(LoRALinear(d, d, rank=4, alpha=4.0), nn.Linear(d, 1))
    run_experiment('LoRA rank=4', lora4)

    # LoRA rank=16
    lora16 = nn.Sequential(LoRALinear(d, d, rank=16, alpha=16.0), nn.Linear(d, 1))
    run_experiment('LoRA rank=16', lora16)
except (NotImplementedError, TypeError):
    print('□ Finish the LoRALinear exercise (section 4) before running this experiment')


## Wrap-up

This section implemented `LoRALinear`: freeze the base linear layer and let only the low-rank patch beside it, `lora_B @ lora_A`, learn the new task. This isn't "rewriting the whole model" — it's adding a small, stable trainable module next to the original; and initializing `B=0` guarantees that training starts out barely disturbing the original model. Next up, L85 carries the same attention projections forward into a cache, to show how a KV cache saves the repeated computation.


## ✏️ Closed-book derivation check cell — LoRA parameter counts

**Rule: collapse every cell above and do the derivation below from memory alone.**

**Problem**: the original weight matrix is $W \in \mathbb{R}^{d_{out} \times d_{in}}$, and LoRA injects $\Delta W = BA$, with $B \in \mathbb{R}^{d_{out} \times r}$ and $A \in \mathbb{R}^{r \times d_{in}}$.

1. Parameters in the original matrix: $|W| = ?$
2. LoRA parameters: $|A| + |B| = ?$
3. Parameter ratio (LoRA / original) = ?
4. If $d_{in} = d_{out} = 768$ and $r = 8$, compute the ratio (to two decimal places)

(Write your derivation here...)

In [ ]:
# Check: parameter-count arithmetic
def lora_savings(d_in, d_out, r):
    original = d_in * d_out
    lora = r * (d_in + d_out)
    ratio = lora / original
    return original, lora, ratio

cases = [(768, 768, 8), (1024, 1024, 16), (768, 3072, 4)]
for d_in, d_out, r in cases:
    orig, lo, ratio = lora_savings(d_in, d_out, r)
    print(f"d_in={d_in} d_out={d_out} r={r}: "
          f"orig={orig:,} lora={lo:,} ratio={ratio:.2%}")

# Concrete check for 768×768, r=8
orig, lo, ratio = lora_savings(768, 768, 8)
assert abs(ratio - 8*(768+768)/(768*768)) < 1e-9
expected = 2*8*768 / (768*768)   # ≈ 2.08%
assert abs(ratio - expected) < 1e-6, f"ratio={ratio:.4f} vs expected={expected:.4f}"
print(f"\n✅ LoRA ratio checked: 768×768, r=8 → uses {ratio:.2%} of the parameters")

In [ ]:
# ✏️ Self-review for this lesson
l84_review = {
    "lora_intuition":            None,  # understand LoRA = frozen W₀ + a parallel low-rank ΔW=B·A? True/False
    "zero_init_rationale":       None,  # understand B=0 init: ΔW=0 at the start, so pretrained ability is intact? True/False
    "lora_linear_impl":          None,  # LoRALinear correct: frozen base, scale=α/r, B zero-initialized? True/False
    "param_savings_formula":     None,  # can derive the saving: d² vs low-rank d·r+r·d, saving (1-2r/d)? True/False
    "whiteboard_passed":         None,  # whiteboard challenge (parameter-count derivation) done? True/False
}

unfilled = [k for k, v in l84_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l84_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L84 fully cleared! On to L85: a KV cache from scratch')

---

→ **Next lesson** · [L85 · A KV cache from scratch](L85_kv_cache.ipynb)

> Next lesson covers **a KV cache from scratch**: updating the key/value cache, O(seq²) → O(seq) per step.